# 01 — Entraînement EfficientNetV2-S + CBAM (version V1)
**EfficientNetV2-S + CBAM multi-échelle** | ISIC 2019 + HAM10000 (dédupliqués) | PH2 en validation externe | Temperature scaling

Premier notebook du projet [skin-lesion-classification](https://github.com/RihemBousbih1/skin-lesion-classification) :
fusion et dédoublonnage des données, split groupé par lésion, entraînement progressif, première calibration.

> **Notes de lecture**
> - Le modèle est un **EfficientNetV2-S** (et non un ResNet50) : les fichiers `resnet50_cbam_*.keras` gardent le nom
>   d'un ancien prototype car les notebooks suivants chargent ces chemins.
> - La loss utilisée est l'**entropie croisée avec label smoothing** ; les focal losses de l'étape 3 sont définies mais non utilisées.
> - Les phases 2a, 2b et 3 ont été **interrompues par des NaN** (float16) : les meilleurs checkpoints ont été rechargés.
>   La phase 3 est reprise et terminée dans `03_phase3.ipynb`.
> - Résultat de cette version (V1) : accuracy test 0.829, macro-F1 0.735, macro AUC 0.969.

## Step 1 — Setup : seeds, GPU, mixed precision, constantes

In [1]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

mixed_precision.set_global_policy("mixed_float16")

# ─── Constantes globales ───────────────────────────────────
IMG_SIZE   = 384
BATCH_SIZE = 32      
EVAL_RESIZE  = int(IMG_SIZE * 1.15)   # resize avant center-crop à l'inférence
NUM_CLASSES  = 7
AUTOTUNE     = tf.data.AUTOTUNE

# Régularisation
LABEL_SMOOTH = 0.1
WEIGHT_DECAY = 1e-4
MIXUP_ALPHA  = 0.1

# Mapping stable (utilisé partout dans le projet)
CLASSES      = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_NAMES_FULL = {
    "akiec": "Actinic keratoses",
    "bcc":   "Basal cell carcinoma",
    "bkl":   "Benign keratosis-like lesions",
    "df":    "Dermatofibroma",
    "nv":    "Melanocytic nevi",
    "mel":   "Melanoma",
    "vasc":  "Vascular lesions",
}

print("TF:", tf.__version__)
print("GPUs:", gpus)
print("Mixed precision:", mixed_precision.global_policy())
print("Mapping:", CLASS_TO_IDX)


strategy = tf.distribute.MirroredStrategy(
    cross_device_ops=tf.distribute.ReductionToOneDevice())
print("Répliques:", strategy.num_replicas_in_sync)   # doit afficher 2

TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision: <DTypePolicy "mixed_float16">
Mapping: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'nv': 4, 'mel': 5, 'vasc': 6}
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Répliques: 2


I0000 00:00:1789733665.970326      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789733665.973284      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
print(IMG_SIZE, strategy.num_replicas_in_sync)   # → 384 2

384 2


## Step 2 — Custom layers CBAM (définis UNE seule fois)

In [3]:
from tensorflow.keras import layers, Model

class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.add = layers.Add()
        self.act = layers.Activation("sigmoid")
        self.mul = layers.Multiply()

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden   = max(channels // self.ratio, 1)
        self.d1  = layers.Dense(hidden, activation="relu",
                                kernel_initializer="he_normal", use_bias=True,
                                dtype="float32")
        self.d2  = layers.Dense(channels,
                                kernel_initializer="he_normal", use_bias=True,
                                dtype="float32")
        self.d1.build((None, 1, 1, channels))
        self.d2.build((None, 1, 1, hidden))
        super().build(input_shape)

    def call(self, x):
        c   = int(x.shape[-1])
        avg = tf.reshape(self.gap(x), (-1, 1, 1, c))
        mx  = tf.reshape(self.gmp(x), (-1, 1, 1, c))
        avg = self.d2(self.d1(avg))
        mx  = self.d2(self.d1(mx))
        return self.mul([x, self.act(self.add([avg, mx]))])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"ratio": self.ratio})
        return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.mul = layers.Multiply()

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding="same",
                                  activation="sigmoid",
                                  kernel_initializer="he_normal", use_bias=False,
                                  dtype="float32")
        self.conv.build((input_shape[0], input_shape[1], input_shape[2], 2))
        super().build(input_shape)

    def call(self, x):
        avg    = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx     = tf.reduce_max(x,  axis=-1, keepdims=True)
        concat = tf.concat([avg, mx], axis=-1)
        return self.mul([x, self.conv(concat)])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"kernel_size": self.kernel_size})
        return cfg


CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention, "SpatialAttention": SpatialAttention}
print("CBAM layers définis.")

CBAM layers définis.


## Step 3 — Losses (focal losses définies pour comparaison, **non utilisées** : l'entraînement utilise l'entropie croisée + label smoothing)

In [4]:
def focal_loss(gamma=2.0, alpha=0.25):
    """Focal loss standard (sans class weights)."""
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce     = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    return loss_fn


def focal_loss_weighted(class_weight_tensor=None, gamma=2.0, alpha=0.25):
    """Focal loss avec pondération par classe (tensor constant)."""
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce     = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1.0 - y_pred, gamma)
        base   = tf.reduce_sum(weight * ce, axis=-1)
        if class_weight_tensor is None:
            return base
        sample_w = tf.reduce_sum(y_true * class_weight_tensor[None, :], axis=-1)
        return sample_w * base
    return loss_fn


print("Losses définies.")

def focal_loss_alpha_per_class(alpha_vec, gamma=2.0):
    """Version correcte : alpha dépend de la classe.
    À n'utiliser QUE si tu veux comparer avec/sans focal dans l'ablation."""
    alpha_vec = tf.constant(alpha_vec, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce     = -y_true * tf.math.log(y_pred)
        w      = alpha_vec[None, :] * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(w * ce, axis=-1)
    return loss_fn

Losses définies.


## Step 4 — Pipeline image tf.data (commun aux 3 datasets)

In [5]:
class NanWatch(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch % 50:
            return
        for v in self.model.variables:
            if not bool(tf.reduce_all(tf.math.is_finite(tf.cast(v, tf.float32)))):
                print(f"\n⚠️  NON-FINI dans « {v.path} » au batch {batch}")
                self.model.stop_training = True
                return

In [6]:
PREPROCESS = lambda x: x        # EfficientNetV2 : preprocessing intégré au modèle

def decode_image(path):
    """Décode jpg/png/bmp selon l'extension."""
    img_bytes = tf.io.read_file(path)
    ext = tf.strings.lower(tf.strings.split(path, ".")[-1])

    def _jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _png(): return tf.image.decode_png(img_bytes,  channels=3)
    def _bmp(): return tf.image.decode_bmp(img_bytes)

    img = tf.case(
        [(tf.equal(ext, "jpg"),  _jpg),
         (tf.equal(ext, "jpeg"), _jpg),
         (tf.equal(ext, "png"),  _png),
         (tf.equal(ext, "bmp"),  _bmp)],
        default=_jpg, exclusive=True
    )
    return tf.ensure_shape(img, [None, None, 3])


# ── NOUVEAU : normalisation d'illuminant (Shades of Gray, p=6) ──
# Neutralise la balance des blancs du dermatoscope. C'est le correctif
# principal pour l'écart de généralisation observé sur PH2.
def shades_of_gray(img, p=6.0):
    """img : float32 en échelle 0-255. Retourne float32 0-255."""
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def augment_train(img):
    """img : float32, 0-255, AVANT resnet_preprocess."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))

    # MODIFIÉ : amplitudes doublées pour couvrir la variabilité inter-appareils
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.30)      # était 0.15
    img = tf.image.random_contrast(img,   0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_saturation(img, 0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_hue(img,        0.08)      # était 0.03
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0

    # cutout à forme statique : masque booléen sur une grille fixe
    h  = tf.random.uniform([], IMG_SIZE//8, IMG_SIZE//4, dtype=tf.int32)
    y0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    yy = tf.range(IMG_SIZE)[:, None]
    xx = tf.range(IMG_SIZE)[None, :]
    inside = (yy >= y0) & (yy < y0 + h) & (xx >= x0) & (xx < x0 + h)
    apply_cut = tf.cast(tf.random.uniform([]) < 0.5, tf.float32)
    keep = 1.0 - apply_cut * tf.cast(inside, tf.float32)[:, :, None]
    img = img * keep
    return tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])


def load_and_preprocess(path, label, training=False):
    img = tf.cast(decode_image(path), tf.float32)

    if training:
        shape = tf.shape(img)
        scale = tf.random.uniform([], 0.7, 1.0)
        h = tf.cast(tf.cast(shape[0], tf.float32) * scale, tf.int32)
        w = tf.cast(tf.cast(shape[1], tf.float32) * scale, tf.int32)
        img = tf.image.random_crop(img, [h, w, 3])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = augment_train(img)
    else:
        # MODIFIÉ : resize + center-crop, au lieu d'un resize direct.
        # L'entraînement voit des crops à 70-100 % ; un resize plein cadre
        # à l'inférence crée un décalage d'échelle systématique.
        img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
        off = (EVAL_RESIZE - IMG_SIZE) // 2
        img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)

    img   = shades_of_gray(img)          # NOUVEAU — train ET inférence
    img   = PREPROCESS(img)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label


def make_dataset(df, training=False, cache=False, shuffle_buffer=4096, batched=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values.astype(str), df["label"].values.astype(np.int32))
    )
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, training=training),
                num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if batched:                              # NOUVEAU : option non-batchée
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


# ── NOUVEAU : mixup ──
def mixup(ds, alpha=MIXUP_ALPHA):
    """À appliquer APRÈS .batch(). Produit des labels mous."""
    def _mix(imgs, labels):
        b   = tf.shape(imgs)[0]
        g1  = tf.random.gamma([b], alpha)
        g2  = tf.random.gamma([b], alpha)
        lam = g1 / (g1 + g2)                  # Beta(alpha, alpha)
        idx = tf.random.shuffle(tf.range(b))
        li  = tf.reshape(lam, [b, 1, 1, 1])
        ll  = tf.reshape(lam, [b, 1])
        return (li * imgs   + (1 - li) * tf.gather(imgs,   idx),
                ll * labels + (1 - ll) * tf.gather(labels, idx))
    return ds.map(_mix, num_parallel_calls=AUTOTUNE)


# ── NOUVEAU : échantillonnage équilibré (remplace la duplication) ──
def balanced_dataset(df, power=0.5):
    """Poids par classe ∝ n^power. power=0.5 (racine) = compromis usuel ;
    power=0 = uniforme strict ; power=1 = distribution naturelle."""
    dss, weights = [], []
    for c in CLASSES:
        sub = df[df["dx"] == c]
        if len(sub) == 0:
            continue
        d = make_dataset(sub, training=True,
                         shuffle_buffer=min(len(sub), 4096), batched=False)
        dss.append(d.repeat())
        weights.append(float(len(sub)) ** power)
    w = np.array(weights) / np.sum(weights)
    ds = tf.data.Dataset.sample_from_datasets(dss, weights=list(w), seed=SEED)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = \
        tf.data.experimental.AutoShardPolicy.DATA
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE).with_options(opts)


print("Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).")

Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).


In [7]:
from sklearn.metrics import f1_score
from tensorflow.keras.optimizers.schedules import CosineDecay


class MacroF1(tf.keras.callbacks.Callback):
    """Calcule val_macro_f1 en fin d'époque. À placer EN PREMIER dans callbacks."""
    def __init__(self, val_ds, y_true):
        super().__init__()
        self.ds, self.y = val_ds, y_true

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = np.argmax(self.model.predict(self.ds, verbose=0), axis=1)
        logs["val_macro_f1"] = f1_score(self.y, p, average="macro")
        print(f"   val_macro_f1: {logs['val_macro_f1']:.4f}")


def make_callbacks(val_ds, y_val, ckpt_path, patience=10):
    return [
        MacroF1(val_ds, y_val),
        NanWatch(),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_macro_f1",
                                           mode="max", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                         patience=patience,
                                         restore_best_weights=True),
    ]


def cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs=2):
    spe = max(n_train // BATCH_SIZE, 1)
    return CosineDecay(initial_learning_rate=peak_lr / 10,
                       decay_steps=spe * n_epochs,
                       warmup_target=peak_lr,
                       warmup_steps=spe * warmup_epochs,
                       alpha=0.01)


print("Callbacks et schedule définis.")
def make_optimizer(peak_lr, n_epochs, n_train, warmup_epochs=2):
    return tf.keras.optimizers.AdamW(
        learning_rate=cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs),
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )

CE_SMOOTH = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

Callbacks et schedule définis.


## Step 5 — Dataset HAM10000

In [8]:
import pandas as pd
from glob import glob

BASE_HAM_DIR = "/kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification"
META_HAM     = os.path.join(BASE_HAM_DIR, "GroundTruth.csv")
IMG_HAM_DIR  = os.path.join(BASE_HAM_DIR, "images")

skin_df = pd.read_csv(META_HAM)

# Recréer dx depuis one-hot
onehot_map = {"MEL": "mel", "NV": "nv", "BCC": "bcc",
              "AKIEC": "akiec", "BKL": "bkl", "DF": "df", "VASC": "vasc"}
onehot_cols = [c for c in onehot_map if c in skin_df.columns]
skin_df["dx"] = skin_df[onehot_cols].idxmax(axis=1).map(onehot_map)

# Chemin image
ham_id_col = "image_id" if "image_id" in skin_df.columns else "image"
img_dict = {os.path.splitext(os.path.basename(p))[0]: p
            for p in glob(os.path.join(IMG_HAM_DIR, "*.jpg"))}
skin_df["path"] = skin_df[ham_id_col].map(img_dict)
skin_df = skin_df.dropna(subset=["path"]).reset_index(drop=True)

# Labels
skin_df["label"]     = skin_df["dx"].map(CLASS_TO_IDX).astype(int)
skin_df["source"]    = "HAM10000"
skin_df["image_uid"] = skin_df[ham_id_col].astype(str)

# ─── Récupérer lesion_id depuis le metadata officiel ──────
META_HAM2_CANDIDATES = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv",
    os.path.join(BASE_HAM_DIR, "HAM10000_metadata.csv"),
]
meta2_path = next((p for p in META_HAM2_CANDIDATES if os.path.exists(p)), None)

if meta2_path:
    meta2 = pd.read_csv(meta2_path)[["image_id", "lesion_id"]]
    meta2["image_id"] = meta2["image_id"].astype(str)
    skin_df = skin_df.merge(meta2, left_on="image_uid", right_on="image_id", how="left")
    n_ok = skin_df["lesion_id"].notna().sum()
    print(f"lesion_id récupérés: {n_ok}/{len(skin_df)}  (source: {meta2_path})")
    skin_df["lesion_id"] = skin_df["lesion_id"].fillna(skin_df["image_uid"])
    print("Lésions uniques:", skin_df["lesion_id"].nunique(),
          "| Images:", len(skin_df),
          f"| ~{len(skin_df)/skin_df['lesion_id'].nunique():.2f} images par lésion")
else:
    skin_df["lesion_id"] = skin_df["image_uid"]
    print("⚠️ HAM10000_metadata.csv introuvable — pas de groupement par lésion.")
    print("   Ajoute le dataset 'kmader/skin-cancer-mnist-ham10000' à ton notebook Kaggle.")

print("\nHAM prêt:", skin_df.shape)
print(skin_df["dx"].value_counts())
print("Exemple chemin:", skin_df["path"].iloc[0])

lesion_id récupérés: 10015/10015  (source: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv)
Lésions uniques: 7470 | Images: 10015 | ~1.34 images par lésion

HAM prêt: (10015, 15)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Exemple chemin: /kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification/images/ISIC_0024306.jpg


## Step 6 — Dataset ISIC2019

In [9]:
BASE_ISIC_DIR  = "/kaggle/input/datasets/cdeotte/jpeg-isic2019-512x512"
META_ISIC      = os.path.join(BASE_ISIC_DIR, "train.csv")
IMG_ISIC_DIR   = os.path.join(BASE_ISIC_DIR, "train")

isic_df = pd.read_csv(META_ISIC)
isic_df["diagnosis"] = isic_df["diagnosis"].str.lower().replace({"ak": "akiec"})

# Garder uniquement les 7 classes communes
isic_df = isic_df[isic_df["diagnosis"].isin(CLASSES)].reset_index(drop=True)

isic_df["dx"]        = isic_df["diagnosis"]
isic_df["path"]      = isic_df["image_name"].map(lambda x: os.path.join(IMG_ISIC_DIR, f"{x}.jpg"))
isic_df["label"]     = isic_df["diagnosis"].map(CLASS_TO_IDX).astype(int)
isic_df["source"]    = "ISIC2019"
isic_df["image_uid"] = isic_df["image_name"].astype(str)
isic_df["lesion_id"] = isic_df["image_uid"]   # ISIC2019 ne fournit pas de lesion_id

print("ISIC df:", isic_df.shape)
print(isic_df["dx"].value_counts())
print("\nExemple chemin:", isic_df["path"].iloc[0])
print("Fichier existe:", os.path.exists(isic_df["path"].iloc[0]))

ISIC df: (24703, 17)
dx
nv       12875
mel       4522
bcc       3323
bkl       2624
akiec      867
vasc       253
df         239
Name: count, dtype: int64

Exemple chemin: /kaggle/input/datasets/cdeotte/jpeg-isic2019-512x512/train/ISIC_0000000.jpg
Fichier existe: True


In [10]:
# ─── Groupement par lésion ────────────────────────────────
import os, pandas as pd

def find_lesion_csv(root="/kaggle/input"):
    out = []
    for dp, _, fs in os.walk(root):
        for f in fs:
            if not f.lower().endswith(".csv"):
                continue
            p = os.path.join(dp, f)
            try:
                cols = set(pd.read_csv(p, nrows=0).columns.str.lower())
            except Exception:
                continue
            if "lesion_id" in cols and ({"image_id", "image"} & cols):
                out.append(p)
    return out

lesion_map = {}
for p in find_lesion_csv():
    m = pd.read_csv(p)
    m.columns = m.columns.str.lower()
    id_col = "image_id" if "image_id" in m.columns else "image"
    m = m[[id_col, "lesion_id"]].dropna()
    lesion_map.update(dict(zip(m[id_col].astype(str), m["lesion_id"].astype(str))))
    print(f"{len(m):6d} paires depuis {p}")

if not lesion_map:
    print("\n⚠️  Aucune métadonnée de lésion. Split au niveau image (fuite résiduelle).")
else:
    print(f"\n{len(lesion_map)} associations image → lésion au total.")

for name, df in [("ISIC2019", isic_df), ("HAM10000", skin_df)]:
    mapped = df["image_uid"].map(lesion_map)
    df["lesion_id"] = ("les_" + mapped).fillna("img_" + df["image_uid"])
    n_dup = len(df) - df["lesion_id"].nunique()
    print(f"{name:9s} : {mapped.notna().sum():5d} groupés | "
          f"{len(df)} images → {df['lesion_id'].nunique()} groupes "
          f"({n_dup} doublons neutralisés)")

 10015 paires depuis /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv

10015 associations image → lésion au total.
ISIC2019  :  9818 groupés | 24703 images → 22215 groupes (2488 doublons neutralisés)
HAM10000  : 10015 groupés | 10015 images → 7470 groupes (2545 doublons neutralisés)


In [11]:
ham_ids  = set(skin_df["image_uid"])
isic_ids = set(isic_df["image_uid"])
overlap  = ham_ids & isic_ids

print(f"Images HAM10000 : {len(ham_ids):6d}")
print(f"Images ISIC2019 : {len(isic_ids):6d}")
print(f"Recouvrement    : {len(overlap):6d}  ({100*len(overlap)/len(ham_ids):.1f}% de HAM)")

if len(overlap) > 0:
    print("\n⚠️ ISIC2019 contient déjà des images de HAM10000 (BCN20000 + HAM + MSK).")
    print("   Exemples:", sorted(overlap)[:5])
    print("   → un split séparé par dataset créerait une fuite train/test.")
else:
    print("\n✅ Aucun recouvrement détecté.")

Images HAM10000 :  10015
Images ISIC2019 :  24703
Recouvrement    :   9818  (98.0% de HAM)

⚠️ ISIC2019 contient déjà des images de HAM10000 (BCN20000 + HAM + MSK).
   Exemples: ['ISIC_0024306', 'ISIC_0024307', 'ISIC_0024308', 'ISIC_0024309', 'ISIC_0024310']
   → un split séparé par dataset créerait une fuite train/test.


In [12]:
from sklearn.model_selection import GroupShuffleSplit

COLS = ["path", "dx", "label", "source", "lesion_id", "image_uid"]

# ISIC est prioritaire ; on n'ajoute de HAM que ce qui n'y est pas déjà
ham_only = skin_df[~skin_df["image_uid"].isin(isic_ids)].copy()
print(f"HAM conservé après retrait du recouvrement: {len(ham_only)} / {len(skin_df)}")

all_df = pd.concat([isic_df[COLS], ham_only[COLS]], ignore_index=True)
all_df = all_df.drop_duplicates(subset=["image_uid"]).reset_index(drop=True)
print(f"Référentiel unique: {len(all_df)} images, {all_df['lesion_id'].nunique()} lésions\n")
print(all_df.groupby(["source", "dx"]).size().unstack(fill_value=0))

# ─── Split groupé par lésion : 70 / 15 / 15 ───────────────
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
i_tv, i_te = next(gss1.split(all_df, groups=all_df["lesion_id"]))
train_val_df, test_df = all_df.iloc[i_tv].copy(), all_df.iloc[i_te].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=SEED)
i_tr, i_va = next(gss2.split(train_val_df, groups=train_val_df["lesion_id"]))
train_df = train_val_df.iloc[i_tr].reset_index(drop=True)
val_df   = train_val_df.iloc[i_va].reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# ─── Vérification anti-fuite (doit afficher 0 partout) ────
print("\n=== Vérification anti-fuite ===")
for a, b, nom in [(train_df, val_df, "train/val"), (train_df, test_df, "train/test"),
                  (val_df, test_df, "val/test")]:
    n_img = len(set(a["image_uid"]) & set(b["image_uid"]))
    n_les = len(set(a["lesion_id"]) & set(b["lesion_id"]))
    flag  = "✅" if (n_img == 0 and n_les == 0) else "❌"
    print(f"{flag} {nom:12s} {n_img} images communes, {n_les} lésions communes")

print(f"\nSplit: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("\nDistribution train:")
print(train_df["dx"].value_counts())
print("\nDistribution test:")
print(test_df["dx"].value_counts())

# ─── Datasets tf.data ─────────────────────────────────────
# Cache mémoire : ~0.6 Mo/image à 224, ~1.8 Mo/image à 384.
# À 384 les deux caches pèsent 13 Go → on désactive.
USE_CACHE = (IMG_SIZE <= 256)

train_ds = make_dataset(train_df, training=True, shuffle_buffer=min(len(train_df), 20000))
val_ds   = make_dataset(val_df,   training=False, cache=USE_CACHE)
test_ds  = make_dataset(test_df,  training=False, cache=USE_CACHE)

mo = IMG_SIZE * IMG_SIZE * 3 * 4 / 1e6
print(f"\nDatasets prêts. Cache mémoire : {'activé' if USE_CACHE else 'désactivé'} "
      f"({mo * (len(val_df) + len(test_df)) / 1000:.1f} Go si activé)")

HAM conservé après retrait du recouvrement: 197 / 10015
Référentiel unique: 24900 images, 22355 lésions

dx        akiec   bcc   bkl   df   mel     nv  vasc
source                                             
HAM10000    197     0     0    0     0      0     0
ISIC2019    867  3323  2624  239  4522  12875   253

=== Vérification anti-fuite ===
✅ train/val    0 images communes, 0 lésions communes
✅ train/test   0 images communes, 0 lésions communes
✅ val/test     0 images communes, 0 lésions communes

Split: train=17455  val=3707  test=3738

Distribution train:
dx
nv       9005
mel      3151
bcc      2369
bkl      1833
akiec     753
df        175
vasc      169
Name: count, dtype: int64

Distribution test:
dx
nv       1976
mel       690
bcc       468
bkl       387
akiec     153
vasc       36
df         28
Name: count, dtype: int64

Datasets prêts. Cache mémoire : désactivé (13.2 Go si activé)


## Step 7 — Dataset PH2

In [13]:
import glob as glob_module  # évite le conflit avec la fonction glob de Step 5

BASE_PH2_DIR  = "/kaggle/input/datasets/spacesurfer/ph2-dataset/PH2Dataset"
PH2_IMG_DIR   = os.path.join(BASE_PH2_DIR, "PH2 Dataset images")
PH2_META_PATH = os.path.join(BASE_PH2_DIR, "PH2_dataset.xlsx")

ph2_paths = glob_module.glob(os.path.join(PH2_IMG_DIR, "*", "*_Dermoscopic_Image", "*.bmp"))
print("Images BMP trouvées:", len(ph2_paths))

ph2_df = pd.DataFrame({"path": ph2_paths})
ph2_df["image_id"] = ph2_df["path"].map(
    lambda p: os.path.basename(os.path.dirname(os.path.dirname(p)))
).str.strip().str.upper()

raw_meta = pd.read_excel(PH2_META_PATH, skiprows=15)
ph2_id_col   = "IMD016" if "IMD016" in raw_meta.columns else raw_meta.columns[0]
ph2_diag_col = 0 if 0 in raw_meta.columns else raw_meta.columns[1]
meta_df = raw_meta[[ph2_id_col, ph2_diag_col]].rename(
    columns={ph2_id_col: "image_id", ph2_diag_col: "diagnosis_code"})
meta_df["image_id"] = meta_df["image_id"].astype(str).str.strip().str.upper()

ph2_df = ph2_df.merge(meta_df, on="image_id", how="inner")

# 0=normal nevi, 1=atypical nevi, 2=melanoma -> mapper sur nos classes
ph2_df["dx"] = ph2_df["diagnosis_code"].map({0: "nv", 1: "nv", 2: "mel"})
ph2_df = ph2_df[ph2_df["dx"].isin(["nv", "mel"])].reset_index(drop=True)
ph2_df["label"]     = ph2_df["dx"].map(CLASS_TO_IDX).astype(int)
ph2_df["source"]    = "PH2"
ph2_df["image_uid"] = ph2_df["image_id"]
ph2_df["lesion_id"] = ph2_df["image_id"]

print("PH2 df:", ph2_df.shape)
print(ph2_df["dx"].value_counts())

# ─── PH2 = jeu de test externe complet ────────────────────
# Jamais utilisé pour l'entraînement : c'est la validation sur données
# indépendantes (autre hôpital, autre dermatoscope).
ph2_ds = make_dataset(ph2_df, training=False)
print(f"\nPH2 (validation externe) prêt: {len(ph2_df)} images, jamais vues à l'entraînement.")

Images BMP trouvées: 200
PH2 df: (197, 8)
dx
nv     145
mel     52
Name: count, dtype: int64

PH2 (validation externe) prêt: 197 images, jamais vues à l'entraînement.


## Step 9 — Construction du modèle EfficientNetV2-S + CBAM multi-échelle

In [14]:
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import BatchNormalization

def build_model(num_classes=NUM_CLASSES, input_shape=(IMG_SIZE, IMG_SIZE, 3),
                use_cbam=True, multi_scale=True, lr=1e-4):
    """use_cbam=False sert à produire la ligne d'ablation « sans CBAM »."""
    base = EfficientNetV2S(weights="imagenet", include_top=False,
                           input_shape=input_shape, include_preprocessing=True)
    def cbam_branch(tensor, tag):
        if use_cbam:
            tensor = ChannelAttention(name=f"cbam_ch_{tag}")(tensor)
            tensor = SpatialAttention(name=f"cbam_sp_{tag}")(tensor)
        return layers.GlobalAveragePooling2D(name=f"gap_{tag}",
                                             dtype="float32")(tensor)

    c5 = cbam_branch(base.output, "c5")

    if multi_scale:
        target  = input_shape[0] // 16
        c4_feat = next(l.output for l in reversed(base.layers)
                       if len(l.output.shape) == 4 and l.output.shape[1] == target)
        
        c4 = cbam_branch(c4_feat, "c4")
        x  = layers.Concatenate(name="multi_scale")([c4, c5])
    else:
        x = c5
        
    x   = layers.LayerNormalization(name="head_ln", dtype="float32")(x)
    x   = layers.Dropout(0.4, dtype="float32")(x)
    out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    return Model(inputs=base.input, outputs=out), base


def unfreeze_last_fraction(base, frac):
    """Remplace le filtrage par 'conv4'/'conv5' : ces noms n'existent pas
    hors ResNet. On dégèle la dernière fraction des couches."""
    cut = int(len(base.layers) * (1 - frac))
    for i, l in enumerate(base.layers):
        l.trainable = (i >= cut)
    for l in base.layers:
        if isinstance(l, BatchNormalization):
            l.trainable = False
    n = sum(1 for l in base.layers if l.trainable)
    print(f"Couches entraînables: {n}/{len(base.layers)}")


with strategy.scope():
    model, base_model = build_model()
print("Modèle construit.")

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Modèle construit.


## Step 10 — Phase 1 : Entraînement backbone gelé (dataset unifié)

In [15]:
# Geler tout le backbone
for layer in base_model.layers:
    layer.trainable = False

EPOCHS_P1 = 15

with strategy.scope():
    model.compile(optimizer=make_optimizer(5e-4, EPOCHS_P1, len(train_df)),
                  loss=CE_SMOOTH, metrics=["accuracy"])

history_phase1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_P1,
    callbacks=make_callbacks(val_ds, val_df["label"].values,
                             "/kaggle/working/resnet50_cbam_phase1.keras",
                             patience=5),
    verbose=1
)
print("Phase 1 terminée.")

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

In [16]:
import shutil, os

src = "/kaggle/working/resnet50_cbam_phase1.keras"
dst = "/kaggle/working/resnet50_cbam_phase1_final.keras"

assert os.path.exists(src), "Checkpoint introuvable — le fit a-t-il tourné ?"
shutil.copy(src, dst)

# Recharge systématique : TerminateOnNaN ne restaure pas les meilleurs poids
model.load_weights(src)

p = model.predict(val_ds, verbose=0)
assert np.isfinite(p).all(), "NaN dans les prédictions — ne continuez pas."
f1 = f1_score(val_df["label"].values, p.argmax(1), average="macro")
print(f"✅ Poids rechargés. val_macro_f1 = {f1:.4f}")


✅ Poids rechargés. val_macro_f1 = 0.4373


In [17]:
import zipfile, os

def save_ckpts(tag):
    zp = f"/kaggle/working/ckpt_{tag}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(os.listdir("/kaggle/working")):
            if f.endswith((".keras", ".npy", ".json")):
                zf.write(f"/kaggle/working/{f}", f)
    print(f"ZIP {tag}: {os.path.getsize(zp)/1e6:.0f} MB — PUBLIEZ EN DATASET MAINTENANT")

## Step 11 — Phase 2 : Fine-tuning progressif (dégel des 30 %, puis 60 % dernières couches du backbone)

In [18]:
from tensorflow.keras.layers import BatchNormalization

# ─── Étape 2a : Débloquer conv5 ───────────────────────────
EPOCHS_P2A = 15
unfreeze_last_fraction(base_model, 0.30)

with strategy.scope():
    model.compile(optimizer=make_optimizer(4e-5, EPOCHS_P2A, len(train_df)),
                  loss=CE_SMOOTH, metrics=["accuracy"])

history_ft1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P2A,
    callbacks=make_callbacks(val_ds, val_df["label"].values,
                             "/kaggle/working/resnet50_cbam_finetune_step1.keras"),
    verbose=1
)
print("Fine-tune step1 terminé.")

Couches entraînables: 123/513
Epoch 1/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step - accuracy: 0.6903 - loss: 1.0821   val_macro_f1: 0.4223
546/546 ━━━━━━━━━━━━━━━━━━━━ 335s 510ms/step - accuracy: 0.6978 - loss: 1.0715 - val_accuracy: 0.7052 - val_loss: 1.0482 - val_macro_f1: 0.4223
Epoch 2/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step - accuracy: 0.7027 - loss: 1.0640   val_macro_f1: 0.4424
546/546 ━━━━━━━━━━━━━━━━━━━━ 244s 445ms/step - accuracy: 0.7043 - loss: 1.0642 - val_accuracy: 0.7143 - val_loss: 1.0282 - val_macro_f1: 0.4424
Epoch 3/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.7133 - loss: 1.0438   val_macro_f1: 0.4777
546/546 ━━━━━━━━━━━━━━━━━━━━ 241s 439ms/step - accuracy: 0.7136 - loss: 1.0413 - val_accuracy: 0.7251 - val_loss: 1.0095 - val_macro_f1: 0.4777
Epoch 4/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step - accuracy: 0.7215 - loss: 1.0265   val_macro_f1: 0.4948
546/546 ━━━━━━━━━━━━━━━━━━━━ 248s 451ms/step - accuracy: 0.7252 - loss: 1.0208 - val_accuracy:

In [19]:
model.load_weights("/kaggle/working/resnet50_cbam_finetune_step1.keras")
p = model.predict(val_ds, verbose=0)
print("finite:", bool(np.isfinite(p).all()))
print("macro_f1:", f1_score(val_df["label"].values, p.argmax(1), average="macro"))

finite: True
macro_f1: 0.6369171210079878


In [20]:
print("Fine-tune step1 terminé.")
save_ckpts("p2a")

Fine-tune step1 terminé.
ZIP p2a: 416 MB — PUBLIEZ EN DATASET MAINTENANT


In [21]:
# ─── Étape 2b : Débloquer conv4 + conv5 ───────────────────
EPOCHS_P2B = 15
unfreeze_last_fraction(base_model, 0.60)

with strategy.scope():
    model.compile(optimizer=make_optimizer(2e-5, EPOCHS_P2B, len(train_df)),
                  loss=CE_SMOOTH, metrics=["accuracy"])
    

history_ft2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P2B,
    callbacks=make_callbacks(val_ds, val_df["label"].values,
                             "/kaggle/working/resnet50_cbam_finetune_step2.keras"),
    verbose=1
)
print("Fine-tune step2 terminé.")

Couches entraînables: 246/513
Epoch 1/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 426ms/step - accuracy: 0.7990 - loss: 0.8755   val_macro_f1: 0.6418
546/546 ━━━━━━━━━━━━━━━━━━━━ 403s 589ms/step - accuracy: 0.7935 - loss: 0.8858 - val_accuracy: 0.7872 - val_loss: 0.8956 - val_macro_f1: 0.6418
Epoch 2/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.8039 - loss: 0.8727   val_macro_f1: 0.6502
546/546 ━━━━━━━━━━━━━━━━━━━━ 286s 522ms/step - accuracy: 0.7990 - loss: 0.8820 - val_accuracy: 0.7955 - val_loss: 0.8925 - val_macro_f1: 0.6502
Epoch 3/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.8060 - loss: 0.8675   val_macro_f1: 0.6622
546/546 ━━━━━━━━━━━━━━━━━━━━ 285s 520ms/step - accuracy: 0.8019 - loss: 0.8741 - val_accuracy: 0.7936 - val_loss: 0.9022 - val_macro_f1: 0.6622
Epoch 4/15
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.8113 - loss: 0.8573   val_macro_f1: 0.6726
546/546 ━━━━━━━━━━━━━━━━━━━━ 282s 513ms/step - accuracy: 0.8131 - loss: 0.8558 - val_accuracy:

## Step 12 — Phase 3 : Fine-tuning final (mixup + échantillonnage équilibré) — interrompu par un NaN à l'époque 5, terminé dans `03_phase3.ipynb`

In [22]:
from tensorflow.keras.layers import BatchNormalization

with strategy.scope():
    model = tf.keras.models.load_model(
        "/kaggle/working/resnet50_cbam_finetune_step2.keras",
        custom_objects=CUSTOM_OBJECTS, compile=False
    )

# ─── SUPPRIMÉ : OVERSAMPLE_MAP et la duplication par sample(replace=True) ───
# Dupliquer df ×4 = 175 images vues 4 fois par époque → mémorisation.
# Cela explique df recall = 0.46 malgré df precision = 0.72.

# ─── NOUVEAU : échantillonnage équilibré + mixup ───
train_ds_bal = mixup(balanced_dataset(train_df, power=0.5))
STEPS_P3     = len(train_df) // BATCH_SIZE

# Contrôle : vérifie la répartition effective sur quelques batches
cnt = np.zeros(NUM_CLASSES)
for _, y in train_ds_bal.take(30):
    cnt += tf.reduce_sum(y, axis=0).numpy()
print("Répartition après rééquilibrage:",
      dict(zip(CLASSES, (cnt / cnt.sum()).round(3))))

EPOCHS_P3 = 40

with strategy.scope():
    for layer in model.layers:
        layer.trainable = True
        if isinstance(layer, BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=make_optimizer(2e-5, EPOCHS_P3, len(train_df)),
                  loss=CE_SMOOTH, metrics=["accuracy"])

history_ft3 = model.fit(
    train_ds_bal,
    steps_per_epoch=STEPS_P3,                    # OBLIGATOIRE (dataset infini)
    validation_data=val_ds, epochs=EPOCHS_P3,
    callbacks=make_callbacks(val_ds, val_df["label"].values,
                             "/kaggle/working/resnet50_cbam_final.keras"),
    verbose=1
)
print("Fine-tuning final terminé.")

Répartition après rééquilibrage: {'akiec': np.float64(0.075), 'bcc': np.float64(0.147), 'bkl': np.float64(0.151), 'df': np.float64(0.056), 'nv': np.float64(0.316), 'mel': np.float64(0.215), 'vasc': np.float64(0.04)}
Epoch 1/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.8021 - loss: 0.9561   val_macro_f1: 0.7245
545/545 ━━━━━━━━━━━━━━━━━━━━ 529s 742ms/step - accuracy: 0.8001 - loss: 0.9568 - val_accuracy: 0.8136 - val_loss: 0.8508 - val_macro_f1: 0.7245
Epoch 2/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 572ms/step - accuracy: 0.7945 - loss: 0.9593   val_macro_f1: 0.7333
545/545 ━━━━━━━━━━━━━━━━━━━━ 373s 684ms/step - accuracy: 0.8018 - loss: 0.9490 - val_accuracy: 0.7904 - val_loss: 0.8909 - val_macro_f1: 0.7333
Epoch 3/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 0s 585ms/step - accuracy: 0.8045 - loss: 0.9396   val_macro_f1: 0.7352
545/545 ━━━━━━━━━━━━━━━━━━━━ 374s 686ms/step - accuracy: 0.8033 - loss: 0.9425 - val_accuracy: 0.8187 - val_loss: 0.8313 - val_macro_f1: 0.7352
Epoch 4/40
545/545 

In [23]:
def predict_tta(model, df, n_rot=4):
    """Moyenne sur 4 rotations × 2 flips.
    Prétraitement identique à load_and_preprocess(training=False)."""
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                img = shades_of_gray(img)          # NOUVEAU — indispensable
                return PREPROCESS(img), y
            ds = (tf.data.Dataset.from_tensor_slices(
                      (df["path"].values.astype(str),
                       df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)

## Step 13 — Évaluation individuelle des checkpoints
(le libellé « final (oversample + focal) » dans les sorties est un ancien nom : il s'agit du checkpoint final, entraîné avec échantillonnage équilibré + mixup)

In [24]:
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.preprocessing import label_binarize


def evaluate_model(model, df, name):
    """Évalue un modèle avec TTA : accuracy, macro-F1, AUC."""
    print(f"\n=== Évaluation : {name} ===")

    preds  = predict_tta(model, df)
    y_pred = np.argmax(preds, axis=1)

    assert len(y_pred) == len(df), f"Désalignement: {len(y_pred)} préds vs {len(df)} lignes"
    y_true = df["label"].values

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"Samples: {len(y_true)}  Accuracy: {acc:.4f}  Macro-F1: {f1:.4f}")

    labels_present = sorted(np.unique(np.concatenate([y_true, y_pred])))
    names = [CLASS_NAMES_FULL.get(IDX_TO_CLASS.get(i, ""), str(i)) for i in labels_present]
    print(classification_report(y_true, y_pred, labels=labels_present,
                                target_names=names, digits=4, zero_division=0))
    try:
        if len(labels_present) > 2:
            yb  = label_binarize(y_true, classes=labels_present)
            auc = roc_auc_score(yb, preds[:, labels_present], average="macro")
        else:
            auc = roc_auc_score(y_true, preds[:, labels_present[1]])
        print(f"Macro AUC (one-vs-rest): {auc:.4f}")
    except Exception as e:
        print("AUC non calculable:", e)

    return preds, y_true, y_pred


# ─── Les 3 checkpoints du pipeline unifié ─────────────────
model_s1 = tf.keras.models.load_model(
    "/kaggle/working/resnet50_cbam_finetune_step1.keras",
    custom_objects=CUSTOM_OBJECTS, compile=False)
model_s2 = tf.keras.models.load_model(
    "/kaggle/working/resnet50_cbam_finetune_step2.keras",
    custom_objects=CUSTOM_OBJECTS, compile=False)
model_final = tf.keras.models.load_model(
    "/kaggle/working/resnet50_cbam_final.keras",
    custom_objects=CUSTOM_OBJECTS, compile=False)

# ─── Évaluation sur le TEST unifié (avec TTA) ─────────────
results = {}
for m, n in [(model_s1, "step1 (conv5)"),
             (model_s2, "step2 (conv4+conv5)"),
             (model_final, "final (oversample + focal)")]:
    preds, y_true, y_pred = evaluate_model(m, test_df, n)
    results[n] = (accuracy_score(y_true, y_pred),
                  f1_score(y_true, y_pred, average="macro", zero_division=0))

print("\n=== Récapitulatif ===")
for n, (a, f) in results.items():
    print(f"{n:30s} Acc={a:.4f}  MacroF1={f:.4f}")


=== Évaluation : step1 (conv5) ===


I0000 00:00:1789745999.278949      70 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Samples: 3738  Accuracy: 0.7935  Macro-F1: 0.6385
                               precision    recall  f1-score   support

            Actinic keratoses     0.6566    0.4248    0.5159       153
         Basal cell carcinoma     0.7527    0.8846    0.8134       468
Benign keratosis-like lesions     0.5915    0.6098    0.6005       387
               Dermatofibroma     0.8571    0.2143    0.3429        28
             Melanocytic nevi     0.8553    0.9271    0.8898      1976
                     Melanoma     0.7592    0.5710    0.6518       690
             Vascular lesions     0.8636    0.5278    0.6552        36

                     accuracy                         0.7935      3738
                    macro avg     0.7623    0.5942    0.6385      3738
                 weighted avg     0.7893    0.7935    0.7847      3738

Macro AUC (one-vs-rest): 0.9554

=== Évaluation : step2 (conv4+conv5) ===
Samples: 3738  Accuracy: 0.8229  Macro-F1: 0.7121
                               precision  

## Step 14 — Prédictions TTA (l'ensemble de checkpoints d'un même entraînement a été abandonné)

In [25]:
def predict_tta(model, df, n_rot=4):
    """Moyenne sur 4 rotations × 2 flips.
    Prétraitement identique à load_and_preprocess(training=False)."""
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                img = shades_of_gray(img)          # NOUVEAU — indispensable
                return PREPROCESS(img), y
            ds = (tf.data.Dataset.from_tensor_slices(
                      (df["path"].values.astype(str),
                       df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)


def get_preds_all_models(df, models):
    """Version TTA : on passe le DataFrame, plus le tf.data.Dataset."""
    return [predict_tta(m, df) for m in models]


print("TTA définie (center-crop + shades-of-gray).")

TTA définie (center-crop + shades-of-gray).


## Step 15 — Temperature Scaling + Thresholds par classe

In [26]:
from scipy.optimize import differential_evolution   # requis par le Step 15

# ─── Ensemble de checkpoints du même run : ABANDONNÉ ──────
MODELS      = [model_final]
MODEL_NAMES = ["final"]
W           = np.array([1.0])


def ensemble_weighted(preds_list, weights):
    """Somme pondérée de probabilités (poids normalisés)."""
    w = np.abs(np.array(weights, dtype=float))
    w = w / (w.sum() + 1e-12)
    return sum(w[i] * preds_list[i] for i in range(len(preds_list)))


# ─── Prédictions VAL (avec TTA) — requises par le Step 15 ─
preds_val_list = get_preds_all_models(val_df, MODELS)
assert len(preds_val_list[0]) == len(val_df), \
    f"Désalignement: {len(preds_val_list[0])} préds vs {len(val_df)} lignes"
y_val_true = val_df["label"].values

acc_solo = accuracy_score(y_val_true, np.argmax(preds_val_list[0], axis=1))
print("Modèle unique retenu (pas d'ensemble intra-run).")
print(f"VAL accuracy (TTA) : {acc_solo:.4f}")

Modèle unique retenu (pas d'ensemble intra-run).
VAL accuracy (TTA) : 0.8279


In [27]:
from scipy.optimize import minimize
from sklearn.metrics import log_loss, recall_score


def apply_temperature(probs, T):
    """Temperature scaling sur des probabilités."""
    probs  = np.clip(probs, 1e-12, 1 - 1e-12)
    logits = np.log(probs)
    scaled = np.exp(logits / T)
    return scaled / scaled.sum(axis=1, keepdims=True)


def find_temperature(probs_val, y_val):
    """Cherche T qui minimise le NLL sur val."""
    def nll(x):
        T = float(x[0])
        if T <= 0:
            return 1e9
        return log_loss(y_val, apply_temperature(probs_val, T),
                        labels=np.arange(probs_val.shape[1]))
    res = minimize(nll, x0=[1.0], bounds=[(0.05, 5.0)], method="L-BFGS-B")
    return float(res.x[0])


# MODIFIÉ : forme multiplicative au lieu du masque + repli.
# probs / thr est un simple repondérage de priors : baisser thr[mel] rend
# la classe mélanome plus facile à atteindre. L'ancienne version à masque
# avait un repli sur argmax qui rendait l'objectif plat par morceaux et
# quasi impossible à optimiser sous contrainte.
def apply_thresholds(probs, thr):
    """Prédit la classe après repondération par classe (thr = diviseurs)."""
    thr = np.clip(np.asarray(thr, dtype=float), 0.05, 5.0)
    return np.argmax(probs / thr[None, :], axis=1)


MEL            = CLASS_TO_IDX["mel"]
CIBLE_SENS_MEL = 0.85     # sensibilité mélanome minimale exigée


def optimize_thresholds(probs_val, y_val):
    """Maximise l'accuracy SOUS CONTRAINTE de sensibilité mélanome >= CIBLE."""
    C = probs_val.shape[1]

    def obj(thr):
        pred = apply_thresholds(probs_val, thr)
        sens = recall_score(y_val, pred, labels=[MEL],
                            average="macro", zero_division=0)
        if sens < CIBLE_SENS_MEL:
            # pénalité dominante : tant que la contrainte n'est pas tenue,
            # l'optimiseur ne cherche qu'à faire monter la sensibilité
            return 10.0 + (CIBLE_SENS_MEL - sens) * 100.0
        return -accuracy_score(y_val, pred)

    res = differential_evolution(obj, [(0.2, 3.0)] * C,
                                 maxiter=80, popsize=10,
                                 polish=False,      # objectif en escalier
                                 seed=SEED)
    return np.clip(res.x, 0.2, 3.0)


# ─── Calibration unique sur le VAL unifié ─────────────────
probs_val = ensemble_weighted(preds_val_list, W)
assert len(probs_val) == len(val_df), \
    f"Désalignement: {len(probs_val)} vs {len(val_df)}"

T = find_temperature(probs_val, y_val_true)
probs_val_T = apply_temperature(probs_val, T)
thr = optimize_thresholds(probs_val_T, y_val_true)

print(f"Temperature T = {T:.4f}")
print(f"Thresholds    = {np.round(thr, 3)}")
print("  " + "  ".join(f"{IDX_TO_CLASS[i]}={thr[i]:.2f}" for i in range(NUM_CLASSES)))

# ─── Effet de la calibration sur VAL ──────────────────────
y_raw_val = np.argmax(probs_val, axis=1)
y_cal_val = apply_thresholds(probs_val_T, thr)

acc_raw = accuracy_score(y_val_true, y_raw_val)
acc_cal = accuracy_score(y_val_true, y_cal_val)
f1_raw  = f1_score(y_val_true, y_raw_val, average="macro", zero_division=0)
f1_cal  = f1_score(y_val_true, y_cal_val, average="macro", zero_division=0)


def sens_spec_mel(y_true, y_pred):
    t, p = (y_true == MEL), (y_pred == MEL)
    return ((t & p).sum() / max(t.sum(), 1),
            (~t & ~p).sum() / max((~t).sum(), 1))


sens_raw, spec_raw = sens_spec_mel(y_val_true, y_raw_val)
sens_cal, spec_cal = sens_spec_mel(y_val_true, y_cal_val)

print("\n--- VAL ---")
print(f"brut     Acc={acc_raw:.4f}  MacroF1={f1_raw:.4f}  "
      f"SensMel={sens_raw:.4f}  SpecMel={spec_raw:.4f}")
print(f"calibré  Acc={acc_cal:.4f}  MacroF1={f1_cal:.4f}  "
      f"SensMel={sens_cal:.4f}  SpecMel={spec_cal:.4f}  "
      f"({100*(acc_cal-acc_raw):+.2f} pts d'accuracy)")

if sens_cal < CIBLE_SENS_MEL:
    print(f"\n⚠️ Contrainte NON tenue : {sens_cal:.4f} < {CIBLE_SENS_MEL}.")
    print("   Le modèle ne sépare pas assez bien mel/nv pour atteindre cette")
    print("   sensibilité. Baisse CIBLE_SENS_MEL à 0.85 ou améliore le modèle.")
else:
    print(f"\n✅ Contrainte tenue : sensibilité mélanome {sens_cal:.4f} "
          f"(coût : {100*(acc_raw-acc_cal):.2f} pts d'accuracy)")

Temperature T = 0.7048
Thresholds    = [1.721 2.032 1.869 2.952 0.654 0.219 1.214]
  akiec=1.72  bcc=2.03  bkl=1.87  df=2.95  nv=0.65  mel=0.22  vasc=1.21

--- VAL ---
brut     Acc=0.8279  MacroF1=0.7431  SensMel=0.7063  SpecMel=0.9475
calibré  Acc=0.7966  MacroF1=0.7027  SensMel=0.8517  SpecMel=0.8506  (-3.13 pts d'accuracy)

✅ Contrainte tenue : sensibilité mélanome 0.8517 (coût : 3.13 pts d'accuracy)


## Step 16 — Évaluation finale sur le TEST

In [28]:
# ─── Prédictions sur le TEST unifié (avec TTA) ────────────
preds_test_list = get_preds_all_models(test_df, MODELS)
probs_test = ensemble_weighted(preds_test_list, W)
assert len(probs_test) == len(test_df), \
    f"Désalignement: {len(probs_test)} vs {len(test_df)}"
y_true = test_df["label"].values

y_raw  = np.argmax(probs_test, axis=1)
probs_test_T = apply_temperature(probs_test, T)
y_cal  = apply_thresholds(probs_test_T, thr)

# ─── Brut vs calibré : le choix se fait sur le VAL ────────
# (variables f1_raw / f1_cal calculées au Step 15)
# La variante calibrée est retenue par choix clinique, pas par métrique :
# elle garantit la sensibilité mélanome. La perte d'accuracy est assumée.
use_cal = True
print("Variante retenue : calibrée (contrainte de sensibilité mélanome)")

y_best  = y_cal if use_cal else y_raw
print(f"Variante retenue (décidée sur VAL) : "
      f"{'calibrée' if use_cal else 'brute (argmax)'}")

print("\n=== TEST : brut vs calibré (les deux, pour information) ===")
for nom, yp in [("argmax brut", y_raw), ("calibré", y_cal)]:
    print(f"  {nom:12s} Acc={accuracy_score(y_true, yp):.4f}  "
          f"MacroF1={f1_score(y_true, yp, average='macro', zero_division=0):.4f}")

# ─── Rapport global ───────────────────────────────────────
target_names = [CLASS_NAMES_FULL[IDX_TO_CLASS[i]] for i in range(NUM_CLASSES)]
print(f"\n=== TEST global ({len(y_true)} images) ===")
print(f"Accuracy: {accuracy_score(y_true, y_best):.4f}  "
      f"Macro-F1: {f1_score(y_true, y_best, average='macro', zero_division=0):.4f}")
print(classification_report(y_true, y_best, labels=np.arange(NUM_CLASSES),
                            target_names=target_names, digits=4, zero_division=0))
print("Matrice de confusion (lignes = vrai, colonnes = prédit):")
print(confusion_matrix(y_true, y_best, labels=np.arange(NUM_CLASSES)))

try:
    yb  = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    auc = roc_auc_score(yb, probs_test_T, average="macro")
    print(f"\nMacro AUC (one-vs-rest): {auc:.4f}")
except Exception as e:
    print("AUC non calculable:", e)

# ─── Sensibilité au mélanome (métrique clinique clé) ──────
idx_mel = CLASS_TO_IDX["mel"]
mel_true = (y_true == idx_mel)
mel_pred = (y_best == idx_mel)
sens = (mel_true & mel_pred).sum() / max(mel_true.sum(), 1)
spec = (~mel_true & ~mel_pred).sum() / max((~mel_true).sum(), 1)
print(f"\nMélanome — Sensibilité: {sens:.4f}  Spécificité: {spec:.4f}  "
      f"({mel_true.sum()} cas)")

# ─── Détail par source ────────────────────────────────────
print("\n=== TEST par source ===")
for src in sorted(test_df["source"].unique()):
    m = (test_df["source"] == src).values
    print(f"{src:10s} ({m.sum():5d} images)  "
          f"Acc={accuracy_score(y_true[m], y_best[m]):.4f}  "
          f"MacroF1={f1_score(y_true[m], y_best[m], average='macro', zero_division=0):.4f}")

Variante retenue : calibrée (contrainte de sensibilité mélanome)
Variante retenue (décidée sur VAL) : calibrée

=== TEST : brut vs calibré (les deux, pour information) ===
  argmax brut  Acc=0.8285  MacroF1=0.7349
  calibré      Acc=0.7865  MacroF1=0.7024

=== TEST global (3738 images) ===
Accuracy: 0.7865  Macro-F1: 0.7024
                               precision    recall  f1-score   support

            Actinic keratoses     0.6522    0.3922    0.4898       153
         Basal cell carcinoma     0.8504    0.8141    0.8319       468
Benign keratosis-like lesions     0.7833    0.4858    0.5997       387
               Dermatofibroma     0.9333    0.5000    0.6512        28
             Melanocytic nevi     0.9136    0.8563    0.8840      1976
                     Melanoma     0.5459    0.8362    0.6606       690
             Vascular lesions     0.8235    0.7778    0.8000        36

                     accuracy                         0.7865      3738
                    macro avg    

## Step 17 — PH2 : validation externe

In [29]:
print("=== PH2 — validation externe (jamais vu à l'entraînement) ===")

probs_ph2 = ensemble_weighted(get_preds_all_models(ph2_df, MODELS), W)
assert len(probs_ph2) == len(ph2_df), f"Désalignement: {len(probs_ph2)} vs {len(ph2_df)}"

idx_nv, idx_mel = CLASS_TO_IDX["nv"], CLASS_TO_IDX["mel"]
y_true_ph2 = ph2_df["label"].values

# 1) Prédiction libre sur 7 classes — le vrai test de transfert
y_free = np.argmax(probs_ph2, axis=1)
print(f"\n7 classes libres : Acc={accuracy_score(y_true_ph2, y_free):.4f}")
print("  Répartition des prédictions:",
      {IDX_TO_CLASS[i]: int((y_free == i).sum()) for i in np.unique(y_free)})

# 2) Argmax restreint à nv/mel (PH2 ne contient que ces 2 classes)
sub = probs_ph2[:, [idx_nv, idx_mel]]
y_pred_ph2 = np.where(sub.argmax(axis=1) == 0, idx_nv, idx_mel)

print(f"\nRestreint nv/mel : Acc={accuracy_score(y_true_ph2, y_pred_ph2):.4f}  "
      f"MacroF1={f1_score(y_true_ph2, y_pred_ph2, average='macro', zero_division=0):.4f}")
print(classification_report(y_true_ph2, y_pred_ph2, labels=[idx_nv, idx_mel],
                            target_names=["Melanocytic nevi", "Melanoma"],
                            digits=4, zero_division=0))
print("Matrice de confusion:")
print(confusion_matrix(y_true_ph2, y_pred_ph2, labels=[idx_nv, idx_mel]))

try:
    p_mel = sub[:, 1] / (sub.sum(axis=1) + 1e-12)
    print(f"\nAUC (mel vs nv): {roc_auc_score((y_true_ph2 == idx_mel).astype(int), p_mel):.4f}")
except Exception as e:
    print("AUC non calculable:", e)

=== PH2 — validation externe (jamais vu à l'entraînement) ===

7 classes libres : Acc=0.8528
  Répartition des prédictions: {'bkl': 6, 'nv': 157, 'mel': 34}

Restreint nv/mel : Acc=0.8782  MacroF1=0.8272
                  precision    recall  f1-score   support

Melanocytic nevi     0.8805    0.9655    0.9211       145
        Melanoma     0.8684    0.6346    0.7333        52

        accuracy                         0.8782       197
       macro avg     0.8745    0.8001    0.8272       197
    weighted avg     0.8773    0.8782    0.8715       197

Matrice de confusion:
[[140   5]
 [ 19  33]]

AUC (mel vs nv): 0.8736


## Step 18 — Sauvegarde des artefacts

In [30]:
import time, json

OUT_DIR = "/kaggle/working"
ts = time.strftime("%Y%m%d-%H%M")

# ─── Artefacts d'inférence (un seul jeu) ──────────────────
np.save(os.path.join(OUT_DIR, "ensemble_weights.npy"), W)
np.save(os.path.join(OUT_DIR, "temperature.npy"),      np.array([T]))
np.save(os.path.join(OUT_DIR, "thresholds.npy"),       thr)

# ─── Mapping des classes ──────────────────────────────────
with open(os.path.join(OUT_DIR, "class_mapping.json"), "w") as f:
    json.dump({"class_to_idx": CLASS_TO_IDX,
               "idx_to_class": {str(k): v for k, v in IDX_TO_CLASS.items()},
               "class_names_full": CLASS_NAMES_FULL}, f, indent=2)

# ─── Config de reproduction ───────────────────────────────
config = {
    "timestamp": ts,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "models": MODEL_NAMES,
    "ensemble_weights": W.tolist(),
    "temperature": float(T),
    "thresholds": thr.tolist(),
    "use_calibration": bool(use_cal),
    "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "ph2_external": len(ph2_df),
    "results": {
        "test_acc":      float(accuracy_score(y_true, y_best)),
        "test_macro_f1": float(f1_score(y_true, y_best, average="macro", zero_division=0)),
        "ph2_acc":       float(accuracy_score(y_true_ph2, y_pred_ph2)),
        "ph2_macro_f1":  float(f1_score(y_true_ph2, y_pred_ph2, average="macro", zero_division=0)),
    },
}
with open(os.path.join(OUT_DIR, "run_config.json"), "w") as f:
    json.dump(config, f, indent=2)

# ─── Splits : rejouer l'évaluation sans réentraîner ───────
split_cols = ["image_uid", "path", "dx", "label", "source", "lesion_id"]
for nom, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df[split_cols].to_csv(os.path.join(OUT_DIR, f"split_{nom}.csv"), index=False)
ph2_df[split_cols].to_csv(os.path.join(OUT_DIR, "split_ph2.csv"), index=False)

# ─── Probabilités brutes : refaire la calibration à froid ─
np.save(os.path.join(OUT_DIR, "probs_val.npy"),  probs_val)
np.save(os.path.join(OUT_DIR, "probs_test.npy"), probs_test)
np.save(os.path.join(OUT_DIR, "probs_ph2.npy"),  probs_ph2)

# ─── Prédictions finales, pour analyse d'erreurs ──────────
pd.DataFrame({
    "image_uid": test_df["image_uid"].values,
    "source":    test_df["source"].values,
    "true":      [IDX_TO_CLASS[i] for i in y_true],
    "pred":      [IDX_TO_CLASS[i] for i in y_best],
    "correct":   (y_true == y_best),
    "confidence": probs_test_T.max(axis=1),
}).to_csv(os.path.join(OUT_DIR, "predictions_test.csv"), index=False)

print("Artefacts sauvegardés dans", OUT_DIR, "\n")
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith((".npy", ".json", ".keras", ".csv")):
        size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
        print(f"  {f:45s} {size:8.2f} MB")

Artefacts sauvegardés dans /kaggle/working 

  class_mapping.json                                0.00 MB
  ensemble_weights.npy                              0.00 MB
  predictions_test.csv                              0.20 MB
  probs_ph2.npy                                     0.01 MB
  probs_test.npy                                    0.21 MB
  probs_val.npy                                     0.21 MB
  resnet50_cbam_final.keras                       417.15 MB
  resnet50_cbam_finetune_step1.keras              261.56 MB
  resnet50_cbam_finetune_step2.keras              368.53 MB
  resnet50_cbam_phase1.keras                       96.46 MB
  resnet50_cbam_phase1_final.keras                 96.46 MB
  run_config.json                                   0.00 MB
  split_ph2.csv                                     0.03 MB
  split_test.csv                                    0.46 MB
  split_train.csv                                   2.15 MB
  split_val.csv                                     0.4

In [31]:
# ─── Ablation : le CBAM sert-il vraiment ? ───
# Relance la passe A complète avec chacune de ces trois configurations,
# et reporte val_macro_f1 + test_macro_f1 dans le tableau ci-dessous.
#
#   build_model(use_cbam=False, multi_scale=False)   → baseline ResNet50 nu
#   build_model(use_cbam=True,  multi_scale=False)   → ton CBAM d'origine
#   build_model(use_cbam=True,  multi_scale=True)    → CBAM multi-échelle
#
# | Configuration          | val macro-F1 | test macro-F1 | test acc |
# |------------------------|--------------|---------------|----------|
# | ResNet50 seul          |              |               |          |
# | + CBAM (conv5)         |              |               |          |
# | + CBAM multi-échelle   |              |               |          |

In [32]:


print("Sens. mel cible | Acc test | MacroF1 | Spéc. mel")
for cible in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    CIBLE_SENS_MEL = cible
    t = optimize_thresholds(probs_val_T, y_val_true)
    yp = apply_thresholds(apply_temperature(probs_test, T), t)
    s, sp = sens_spec_mel(y_true, yp)
    print(f"     {cible:.2f}       | {accuracy_score(y_true, yp):.4f}  | "
          f"{f1_score(y_true, yp, average='macro', zero_division=0):.4f}  | {sp:.4f}")
CIBLE_SENS_MEL = 0.85

Sens. mel cible | Acc test | MacroF1 | Spéc. mel
     0.70       | 0.8264  | 0.7476  | 0.9357
     0.75       | 0.8224  | 0.7341  | 0.9199
     0.80       | 0.8068  | 0.7293  | 0.8901
     0.85       | 0.7865  | 0.7024  | 0.8425
     0.90       | 0.7552  | 0.6912  | 0.7907
     0.95       | 0.6964  | 0.6461  | 0.7014
